# ⚡ Meter Digit Detector
### YOLOv8n trained on Number Detection Dataset — detects digits 0–9 on meter displays

**Before running any cell:**
> Runtime → Change Runtime Type → **T4 GPU** → Save

**Resume support:** If your session disconnects, just re-run cells 1 → 2 → 3 → then jump to **Cell 5B** (Resume Training). Your progress is saved to Google Drive automatically every 10 epochs.

---
| Cell | What it does | Run order |
|------|--------------|------|
| 1 | Check GPU | Always |
| 2 | Install libraries | Always |
| 3 | Mount Google Drive + setup paths | Always |
| 4 | Download dataset from Roboflow | First time only |
| 5A | **Start fresh training** | First time only |
| 5B | **Resume training after disconnect** | After disconnect |
| 6 | View training curves | After training |
| 7 | Validate — get mAP score | After training |
| 8 | Test on your photo | After training |
| 9 | Export final model to Drive | After training |

## Cell 1 — Check GPU

In [ ]:
!nvidia-smi
# ✅ Must show Tesla T4 with ~15GB memory.
# If not: Runtime > Change Runtime Type > T4 GPU > Save, then re-run.

## Cell 2 — Install Libraries

In [ ]:
!pip install ultralytics roboflow -q

import ultralytics
print(f'✅ Ultralytics {ultralytics.__version__} installed!')
print('✅ Roboflow installed!')

## Cell 3 — Mount Google Drive & Setup Paths
> **Run this every time** — even after a disconnect. Drive stores your checkpoints.

In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# ── All paths defined once here ─────────────────────────────────────────────
PROJECT_NAME   = 'digit_detector_v1'          # folder name used everywhere
DRIVE_DIR      = f'/content/drive/MyDrive/{PROJECT_NAME}'
LOCAL_RUNS_DIR = f'/content/runs/detect/{PROJECT_NAME}'
DRIVE_WEIGHTS  = f'{DRIVE_DIR}/weights'
DRIVE_CKPT     = f'{DRIVE_DIR}/checkpoints'

# Create Drive folders if they don't exist
for d in [DRIVE_DIR, DRIVE_WEIGHTS, DRIVE_CKPT]:
    os.makedirs(d, exist_ok=True)

print('✅ Google Drive mounted!')
print(f'   Drive folder : {DRIVE_DIR}')
print(f'   Checkpoints  : {DRIVE_CKPT}')
print(f'   Weights      : {DRIVE_WEIGHTS}')

# ── Check if a previous checkpoint exists on Drive ──────────────────────────
last_pt_drive = f'{DRIVE_WEIGHTS}/last.pt'
best_pt_drive = f'{DRIVE_WEIGHTS}/best.pt'

if os.path.exists(last_pt_drive):
    print(f'\n🔄 CHECKPOINT FOUND on Drive: {last_pt_drive}')
    print('   → Skip Cell 5A. Run Cell 5B to RESUME training.')
else:
    print('\n🆕 No checkpoint found — run Cell 4 then Cell 5A to start fresh.')

## Cell 4 — Download Dataset from Roboflow
> **Run once.** If you disconnected and reconnected, run this again (dataset lives in /content which resets).

In [ ]:
from roboflow import Roboflow
import os, yaml

rf      = Roboflow(api_key="0leDKPV6myoVoduqhsIy")
project = rf.workspace("aipoweredsmartmetermanagement").project("number-detection-for-v9-ykyao")
version = project.version(1)
dataset = version.download("yolov8")

# ── Fix data.yaml to use absolute paths ──────────────────────────────────────
DATA_YAML = os.path.join(dataset.location, 'data.yaml')

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

data['train'] = os.path.join(dataset.location, 'train', 'images')
data['val']   = os.path.join(dataset.location, 'valid', 'images')
test_dir      = os.path.join(dataset.location, 'test',  'images')
if os.path.exists(test_dir):
    data['test'] = test_dir

with open(DATA_YAML, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

# ── Save DATA_YAML path to Drive so resume cell can find it ──────────────────
with open(f'{DRIVE_DIR}/data_yaml_path.txt', 'w') as f:
    f.write(DATA_YAML)

# ── Report ───────────────────────────────────────────────────────────────────
train_n = len(os.listdir(data['train']))
val_n   = len(os.listdir(data['val']))

print(f'\n✅ Dataset ready!')
print(f'   Train  : {train_n} images')
print(f'   Val    : {val_n} images')
print(f'   Classes: {data["nc"]} → {data["names"]}')
print(f'   YAML   : {DATA_YAML}')
print(f'\n→ Now run Cell 5A (fresh start) or Cell 5B (resume).')

## Cell 4B — Preview Annotations (optional but recommended)
> Verify the digit bounding boxes look correct before training.

In [ ]:
import cv2, os, random, yaml
import matplotlib.pyplot as plt

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

img_dir  = data['train']
lbl_dir  = img_dir.replace('images', 'labels')
all_imgs = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
sample   = random.sample(all_imgs, min(6, len(all_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

# Each digit class gets a distinct color
COLORS = [
    (255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),
    (0,255,255),(128,0,255),(255,128,0),(0,128,255),(128,255,0)
]

for idx, fname in enumerate(sample):
    img  = cv2.imread(os.path.join(img_dir, fname))
    img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    lbl  = os.path.join(lbl_dir, os.path.splitext(fname)[0] + '.txt')

    if os.path.exists(lbl):
        with open(lbl) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    cls, cx, cy, bw, bh = map(float, parts)
                    x1 = int((cx-bw/2)*w); y1 = int((cy-bh/2)*h)
                    x2 = int((cx+bw/2)*w); y2 = int((cy+bh/2)*h)
                    color = COLORS[int(cls) % len(COLORS)]
                    cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                    name = data['names'][int(cls)] if int(cls) < len(data['names']) else str(int(cls))
                    cv2.putText(img, str(name), (x1, max(y1-6, 0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

    axes[idx].imshow(img)
    axes[idx].set_title(fname, fontsize=8)
    axes[idx].axis('off')

plt.suptitle('Digit Annotations — each box should tightly surround one digit', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ If each digit has its own box → run Cell 5A')
print('❌ If boxes look wrong → fix labels on Roboflow first')

## Cell 5A — 🆕 Start Fresh Training
> **Run only on first time.** If you already trained and disconnected, skip to Cell 5B.
>
> 1,100 images × 100 epochs ≈ **25–40 minutes** on T4 GPU
>
> Checkpoints are auto-saved to Google Drive every 10 epochs.

In [ ]:
from ultralytics import YOLO
from google.colab import drive
import os, shutil, threading, time

# ── Model choice: yolov8n (nano) ─────────────────────────────────────────────
# For 1,100 images and small digit boxes, nano beats small:
#   • Faster training → more epochs in same time
#   • Less overfitting risk on small dataset
#   • Still achieves mAP50 > 0.90 on digit detection tasks
model = YOLO('yolov8n.pt')

# ── Auto-backup thread: copies weights to Drive every 10 min ─────────────────
def auto_backup(interval_sec=600):
    while True:
        time.sleep(interval_sec)
        src_last = f'{LOCAL_RUNS_DIR}/weights/last.pt'
        src_best = f'{LOCAL_RUNS_DIR}/weights/best.pt'
        try:
            if os.path.exists(src_last):
                shutil.copy(src_last, f'{DRIVE_WEIGHTS}/last.pt')
            if os.path.exists(src_best):
                shutil.copy(src_best, f'{DRIVE_WEIGHTS}/best.pt')
            print(f'💾 Auto-backup saved to Drive at {time.strftime("%H:%M:%S")}')
        except Exception as e:
            print(f'⚠️  Backup failed: {e}')

backup_thread = threading.Thread(target=auto_backup, daemon=True)
backup_thread.start()
print('💾 Auto-backup thread started (saves every 10 minutes)')

# ── Training ─────────────────────────────────────────────────────────────────
print('🚀 Starting training...')
model.train(
    data      = DATA_YAML,
    epochs    = 100,          # 100 is correct for 1,100-image digit dataset
    imgsz     = 640,
    batch     = 16,           # T4 handles 16 comfortably; drop to 8 if OOM
    patience  = 25,           # early-stop if no improvement for 25 epochs
    device    = 0,
    workers   = 4,
    name      = PROJECT_NAME,
    exist_ok  = True,
    save      = True,         # save best.pt and last.pt
    save_period = 10,         # also save epoch checkpoints every 10 epochs

    # ── Augmentation tuned for small digit objects ────────────────────────
    # Less aggressive than meter detector — digits are small, flipping breaks them
    hsv_h     = 0.010,        # slight hue shift
    hsv_s     = 0.4,          # saturation variation
    hsv_v     = 0.3,          # brightness variation (meter lighting varies)
    degrees   = 5.0,          # tiny rotation — digits aren't perfectly upright
    translate = 0.1,          # position shift
    scale     = 0.4,          # zoom — handles different meter sizes
    fliplr    = 0.0,          # NO horizontal flip — '6' becomes '9', breaks classes
    flipud    = 0.0,          # NO vertical flip — same reason
    mosaic    = 0.8,          # mosaic helps multi-digit generalization
    copy_paste= 0.1,          # paste random digits into images — great for small dataset
    mixup     = 0.0,          # off — mixup confuses digit boundaries
)

# ── Final backup to Drive ─────────────────────────────────────────────────────
print('\n💾 Saving final weights to Google Drive...')
for fname in ['best.pt', 'last.pt']:
    src = f'{LOCAL_RUNS_DIR}/weights/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_WEIGHTS}/{fname}')
        print(f'   ✅ {fname} → {DRIVE_WEIGHTS}/{fname}')

# Copy epoch checkpoints
ckpt_dir = f'{LOCAL_RUNS_DIR}/weights'
for f in os.listdir(ckpt_dir):
    if f.startswith('epoch') and f.endswith('.pt'):
        shutil.copy(f'{ckpt_dir}/{f}', f'{DRIVE_CKPT}/{f}')

print('\n✅ Training complete!')
print(f'   Best weights → {DRIVE_WEIGHTS}/best.pt')
print('   → Now run Cell 6 to view results')

## Cell 5B — 🔄 Resume Training After Disconnect
> **Run this instead of 5A** if your session disconnected mid-training.
>
> Prerequisites: Run cells 1 → 2 → 3 → 4 first, then come here.

In [ ]:
from ultralytics import YOLO
import os, shutil, threading, time

last_pt_drive = f'{DRIVE_WEIGHTS}/last.pt'
last_pt_local = f'{LOCAL_RUNS_DIR}/weights/last.pt'

# ── Restore checkpoint from Drive to local ────────────────────────────────────
if not os.path.exists(last_pt_drive):
    print('❌ No checkpoint found on Drive!')
    print('   → Run Cell 5A instead (fresh start).')
    raise FileNotFoundError(f'Expected: {last_pt_drive}')

os.makedirs(f'{LOCAL_RUNS_DIR}/weights', exist_ok=True)
shutil.copy(last_pt_drive, last_pt_local)
print(f'✅ Checkpoint restored from Drive → {last_pt_local}')

# Get which epoch we were on
model_check = YOLO(last_pt_local)
start_epoch = getattr(model_check, 'ckpt', {}).get('epoch', '?')
print(f'   Resuming from epoch: {start_epoch}')

# ── Auto-backup thread ────────────────────────────────────────────────────────
def auto_backup(interval_sec=600):
    while True:
        time.sleep(interval_sec)
        try:
            for fname in ['last.pt', 'best.pt']:
                src = f'{LOCAL_RUNS_DIR}/weights/{fname}'
                if os.path.exists(src):
                    shutil.copy(src, f'{DRIVE_WEIGHTS}/{fname}')
            print(f'💾 Auto-backup saved to Drive at {time.strftime("%H:%M:%S")}')
        except Exception as e:
            print(f'⚠️  Backup failed: {e}')

backup_thread = threading.Thread(target=auto_backup, daemon=True)
backup_thread.start()
print('💾 Auto-backup thread started (saves every 10 minutes)')

# ── Resume training ───────────────────────────────────────────────────────────
print('\n🔄 Resuming training from last checkpoint...')
model = YOLO(last_pt_local)
model.train(resume=True)

# ── Final save to Drive ───────────────────────────────────────────────────────
print('\n💾 Saving final weights to Google Drive...')
for fname in ['best.pt', 'last.pt']:
    src = f'{LOCAL_RUNS_DIR}/weights/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE_WEIGHTS}/{fname}')
        print(f'   ✅ {fname} saved to Drive')

print('\n✅ Training resumed and complete!')
print('   → Run Cell 6 to view results')

## Cell 6 — View Training Results

In [ ]:
from IPython.display import Image as IPImage, display
import os

base = LOCAL_RUNS_DIR

for fpath, title in [
    (f'{base}/results.png',          'Training Curves (loss + mAP over epochs)'),
    (f'{base}/confusion_matrix.png', 'Confusion Matrix (which digits get confused)'),
    (f'{base}/val_batch0_pred.jpg',  'Sample Predictions on Validation Set'),
]:
    if os.path.exists(fpath):
        print(f'\n📊 {title}:')
        display(IPImage(fpath, width=900))
    else:
        print(f'⚠️  Not found: {fpath}')

print('\n💡 Tip: In the confusion matrix, look for confusion between 1↔7 and 6↔9.')
print('   If those are high, your dataset may need more variety for those digits.')

## Cell 7 — Validate & Get mAP Score

In [ ]:
from ultralytics import YOLO
import yaml

# Load best weights — try local first, then Drive
BEST_LOCAL = f'{LOCAL_RUNS_DIR}/weights/best.pt'
BEST_DRIVE = f'{DRIVE_WEIGHTS}/best.pt'

best_pt = BEST_LOCAL if os.path.exists(BEST_LOCAL) else BEST_DRIVE
print(f'Loading weights from: {best_pt}')

model   = YOLO(best_pt)
metrics = model.val(data=DATA_YAML, device=0)

map50   = metrics.box.map50
map5095 = metrics.box.map
prec    = metrics.box.mp
rec     = metrics.box.mr

# Per-class results
with open(DATA_YAML) as f:
    class_names = yaml.safe_load(f)['names']

print('\n' + '='*50)
print('  DIGIT DETECTION — VALIDATION RESULTS')
print('='*50)
print(f'  mAP@0.50      : {map50:.4f}')
print(f'  mAP@0.50:0.95 : {map5095:.4f}')
print(f'  Precision     : {prec:.4f}')
print(f'  Recall        : {rec:.4f}')
print('='*50)

# Per-class AP
if hasattr(metrics.box, 'ap_class_index') and metrics.box.ap_class_index is not None:
    print('\n  Per-class mAP@0.50:')
    ap_per_class = metrics.box.ap50
    for i, ap in enumerate(ap_per_class):
        cls_name = class_names[i] if i < len(class_names) else str(i)
        bar = '█' * int(ap * 20)
        print(f'    {str(cls_name):>4}: {bar:<20} {ap:.3f}')

print()
if map50 >= 0.90:
    print('✅ EXCELLENT — model is production ready!')
elif map50 >= 0.80:
    print('✅ GOOD — will work well for digit detection')
elif map50 >= 0.65:
    print('⚠️  DECENT — functional, but some digits may be misread')
else:
    print('❌ LOW — check annotation quality on Roboflow, especially for rare digits')

## Cell 8 — Test on Your Meter Photo
> Upload a cropped meter display image to see which digits are detected.

In [ ]:
from google.colab import files
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import cv2, numpy as np, matplotlib.pyplot as plt, os, yaml

BEST_LOCAL = f'{LOCAL_RUNS_DIR}/weights/best.pt'
BEST_DRIVE = f'{DRIVE_WEIGHTS}/best.pt'
best_pt    = BEST_LOCAL if os.path.exists(BEST_LOCAL) else BEST_DRIVE

model = YOLO(best_pt)
print(f'✅ Model loaded from: {best_pt}')

with open(DATA_YAML) as f:
    class_names = yaml.safe_load(f)['names']

print('📸 Upload a meter photo (full meter or cropped digit area):')
uploaded = files.upload()
img_path = list(uploaded.keys())[0]

# Run inference
results = model(img_path, conf=0.25, verbose=False)
boxes   = results[0].boxes

# Draw results
img     = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
h, w    = img.shape[:2]

COLORS = [
    (255,80,80),(80,255,80),(80,80,255),(255,255,80),(255,80,255),
    (80,255,255),(180,80,255),(255,160,80),(80,160,255),(160,255,80)
]

detected_digits = []

if boxes and len(boxes) > 0:
    # Sort by x-position (left to right) to get reading order
    box_data = []
    for b in boxes:
        x1,y1,x2,y2 = map(int, b.xyxy[0].tolist())
        cls  = int(b.cls[0])
        conf = float(b.conf[0])
        cx   = (x1 + x2) / 2
        box_data.append((cx, x1, y1, x2, y2, cls, conf))

    box_data.sort(key=lambda x: x[0])  # sort left → right

    for _, x1, y1, x2, y2, cls, conf in box_data:
        name  = str(class_names[cls]) if cls < len(class_names) else str(cls)
        color = COLORS[cls % len(COLORS)]
        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), color, 3)
        label = f'{name} {conf:.2f}'
        cv2.putText(img_rgb, label, (x1, max(y1-8, 0)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        detected_digits.append((name, conf))

    reading = ''.join([d[0] for d in detected_digits])
else:
    reading = ''

plt.figure(figsize=(14, 8))
plt.imshow(img_rgb)
plt.title(f'Detected Reading (left→right): {reading}',
          fontsize=16, fontweight='bold',
          color='green' if reading else 'red')
plt.axis('off')
plt.tight_layout()
plt.show()

print('\n' + '='*40)
print(f'  METER READING : {reading if reading else "NO DIGITS FOUND"}')
print('='*40)
if detected_digits:
    print('  Digit  Confidence')
    for d, c in detected_digits:
        print(f'    {d}       {c:.3f}')
else:
    print('  No digits detected at conf=0.25')
    print('  Try: model(img_path, conf=0.10) in a new cell')

## Cell 9 — Export Final Model to Google Drive
> Run this after training is complete. Saves everything you need.

In [ ]:
import shutil, os

# Copy weights
for fname in ['best.pt', 'last.pt']:
    src = f'{LOCAL_RUNS_DIR}/weights/{fname}'
    dst = f'{DRIVE_WEIGHTS}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ {fname} → {dst}')

# Copy training results (curves, confusion matrix)
results_dir = f'{DRIVE_DIR}/results'
os.makedirs(results_dir, exist_ok=True)
for fname in ['results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg',
              'val_batch1_pred.jpg', 'results.csv']:
    src = f'{LOCAL_RUNS_DIR}/{fname}'
    if os.path.exists(src):
        shutil.copy(src, f'{results_dir}/{fname}')
        print(f'✅ {fname} saved')

# Export to ONNX (optional — for deployment without PyTorch)
from ultralytics import YOLO
best_pt = f'{LOCAL_RUNS_DIR}/weights/best.pt'
if os.path.exists(best_pt):
    print('\n📦 Exporting to ONNX format...')
    m = YOLO(best_pt)
    m.export(format='onnx', imgsz=640)
    onnx_src = best_pt.replace('.pt', '.onnx')
    if os.path.exists(onnx_src):
        shutil.copy(onnx_src, f'{DRIVE_WEIGHTS}/best.onnx')
        print(f'✅ best.onnx saved to Drive')

print(f'\n🎉 All done! Everything saved to:')
print(f'   {DRIVE_DIR}')
print(f'\nTo use the model later:')
print(f'   from ultralytics import YOLO')
print(f'   model = YOLO("{DRIVE_WEIGHTS}/best.pt")')
print(f'   results = model("your_image.jpg", conf=0.25)')

---
## 📋 Quick Reference

| Cell | What it does | Time |
|------|--------------|------|
| 1 | Check GPU | 5s |
| 2 | Install libraries | 1 min |
| 3 | Mount Drive + setup paths | 30s |
| 4 | Download digit dataset | 2–5 min |
| 4B | Preview annotations | 10s |
| 5A | **Start fresh training** | 25–40 min |
| 5B | **Resume after disconnect** | remaining time |
| 6 | View training curves | 10s |
| 7 | Validate — get mAP | 1–2 min |
| 8 | Test on your photo | 30s |
| 9 | Export to Drive | 2 min |

## 🔄 Disconnect Recovery — Step by Step

If your Colab session disconnects:
1. **Runtime → Connect** (reconnect to a new session)
2. Run **Cell 1** → check GPU
3. Run **Cell 2** → reinstall libraries
4. Run **Cell 3** → remount Drive (your checkpoint is safe here)
5. Run **Cell 4** → re-download dataset (Colab `/content` resets)
6. Run **Cell 5B** → resume from last checkpoint ✅

## ⚠️ Common Errors

| Error | Fix |
|-------|-----|
| `CUDA out of memory` | Change `batch=16` to `batch=8` in Cell 5A |
| `FileNotFoundError last.pt` | No checkpoint yet — run Cell 5A instead of 5B |
| `No labels found` | Re-run Cell 4 — paths reset after disconnect |
| Low mAP on digit '1' or '7' | Dataset may have too few examples of those digits — check Roboflow class balance |
| Digits read in wrong order | Left-to-right sort in Cell 8 handles this automatically |

## 💡 Why These Settings?

| Setting | Value | Reason |
|---------|-------|--------|
| Model | yolov8n | Nano fits 1,100 images; less overfitting than 's' |
| Epochs | 100 | Small dataset needs more passes; early-stop prevents waste |
| patience | 25 | Generous — digit mAP can improve slowly in later epochs |
| fliplr | 0.0 | 6↔9 and other mirror ambiguities break digit classes |
| copy_paste | 0.1 | Pastes random digits into scenes — helps small dataset |
| save_period | 10 | Saves epoch10, epoch20 … checkpoints for safety |